# 23 — Training Strategy Closure

## Objetivo

Cerrar formalmente la fase de auditoría de datasets públicos y decidir qué fuentes pueden utilizarse para:

- entrenamiento Nielsen directo,
- experimentos proxy,
- validación técnica del pipeline,
- referencia de etiquetas,
- o únicamente apoyo auxiliar.

Este notebook evita seguir forzando fuentes incompatibles y define el siguiente paso metodológico: construir un conjunto **silver** propio donde las X canónicas sean capturadas por la extensión y las Y Nielsen provengan de evaluación LLM explícita y separada.


## 1. Configuración

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 400)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)


## 2. Cargar resultados consolidados

In [13]:
def read_if_exists(path):

    if not path.exists():
        print(
            "FALTA:",
            path.name
        )
        return None

    return pd.read_csv(
        path
    )


uicrit_rico = read_if_exists(
    INTERIM_DIR
    / "uicrit_rico_proxy_dataset.csv"
)

catching = read_if_exists(
    INTERIM_DIR
    / "catching_static_validation_dataset.csv"
)

vt_targets = read_if_exists(
    INTERIM_DIR
    / "vt_recovered_candidate_targets.csv"
)

vt_decision = read_if_exists(
    REPORTS_DIR
    / "vt_recovery_source_decision.csv"
)

dashboards_decision = read_if_exists(
    REPORTS_DIR
    / "dashboards_join_decision.csv"
)

compatibility = read_if_exists(
    REPORTS_DIR
    / "feature_compatibility_matrix.csv"
)

print(
    "UICrit/RICO:",
    None
    if uicrit_rico is None
    else uicrit_rico.shape
)

print(
    "Catching:",
    None
    if catching is None
    else catching.shape
)

print(
    "29042675 targets:",
    None
    if vt_targets is None
    else vt_targets.shape
)


UICrit/RICO: (1000, 23)
Catching: (2, 23)
29042675 targets: (15, 5)


## 3. Verificar diversidad de clases directas

In [14]:
TARGETS = [
    "H1",
    "H4",
    "H5",
    "H6"
]

class_audit_records = []

# Catching
if catching is not None:

    for target in TARGETS:

        col = (
            target
            +
            "_violation"
        )

        values = pd.to_numeric(
            catching[
                col
            ],
            errors="coerce"
        )

        class_audit_records.append({
            "source_dataset":
                "Catching_UX_Flaws",

            "target":
                target,

            "n_rows":
                len(
                    catching
                ),

            "n_labeled":
                int(
                    values.notna().sum()
                ),

            "n_0":
                int(
                    (
                        values
                        ==
                        0
                    ).sum()
                ),

            "n_1":
                int(
                    (
                        values
                        ==
                        1
                    ).sum()
                ),

            "n_classes":
                int(
                    values
                    .dropna()
                    .nunique()
                ),

            "label_provenance":
                "LLM_DIRECT"
        })

# 29042675
if vt_targets is not None:

    for target in TARGETS:

        col = (
            target
            +
            "_candidate"
        )

        if col not in vt_targets.columns:
            continue

        values = pd.to_numeric(
            vt_targets[
                col
            ],
            errors="coerce"
        )

        class_audit_records.append({
            "source_dataset":
                "29042675",

            "target":
                target,

            "n_rows":
                len(
                    vt_targets
                ),

            "n_labeled":
                int(
                    values.notna().sum()
                ),

            "n_0":
                int(
                    (
                        values
                        ==
                        0
                    ).sum()
                ),

            "n_1":
                int(
                    (
                        values
                        ==
                        1
                    ).sum()
                ),

            "n_classes":
                int(
                    values
                    .dropna()
                    .nunique()
                ),

            "label_provenance":
                "LLM_DIRECT_CANDIDATE"
        })

direct_class_audit = pd.DataFrame(
    class_audit_records
)

display(
    direct_class_audit
)


,source_dataset,target,n_rows,n_labeled,n_0,n_1,n_classes,label_provenance
0,Catching_UX_Flaws,H1,2,2,0,2,1,LLM_DIRECT
1,Catching_UX_Flaws,H4,2,2,0,2,1,LLM_DIRECT
2,Catching_UX_Flaws,H5,2,2,0,2,1,LLM_DIRECT
3,Catching_UX_Flaws,H6,2,2,0,2,1,LLM_DIRECT
4,29042675,H1,15,9,0,9,1,LLM_DIRECT_CANDIDATE
5,29042675,H4,15,12,0,12,1,LLM_DIRECT_CANDIDATE
6,29042675,H5,15,6,0,6,1,LLM_DIRECT_CANDIDATE
7,29042675,H6,15,15,0,15,1,LLM_DIRECT_CANDIDATE


## 4. Criterio mínimo de trainability

Un clasificador binario supervisado no se considera entrenable si:

- solo existe una clase observada;
- la X no pertenece a la misma instancia que la Y;
- la X es únicamente un proxy de otra plataforma;
- o la fuente tiene un número de grupos demasiado pequeño para separación train/test.


In [15]:
direct_class_audit[
    "has_both_classes"
] = (
    direct_class_audit[
        "n_classes"
    ]
    >=
    2
)

direct_class_audit[
    "binary_training_ready"
] = False

display(
    direct_class_audit
)


,source_dataset,target,n_rows,n_labeled,n_0,n_1,n_classes,label_provenance,has_both_classes,binary_training_ready
0,Catching_UX_Flaws,H1,2,2,0,2,1,LLM_DIRECT,False,False
1,Catching_UX_Flaws,H4,2,2,0,2,1,LLM_DIRECT,False,False
2,Catching_UX_Flaws,H5,2,2,0,2,1,LLM_DIRECT,False,False
3,Catching_UX_Flaws,H6,2,2,0,2,1,LLM_DIRECT,False,False
4,29042675,H1,15,9,0,9,1,LLM_DIRECT_CANDIDATE,False,False
5,29042675,H4,15,12,0,12,1,LLM_DIRECT_CANDIDATE,False,False
6,29042675,H5,15,6,0,6,1,LLM_DIRECT_CANDIDATE,False,False
7,29042675,H6,15,15,0,15,1,LLM_DIRECT_CANDIDATE,False,False


## 5. Roles finales de las fuentes auditadas

In [16]:
source_role_final = pd.DataFrame([
    {
        "source_dataset":
            "UICrit_RICO",

        "x_status":
            "AVAILABLE_PLATFORM_SPECIFIC",

        "y_status":
            "RATING_PROXY",

        "same_instance_xy":
            True,

        "canonical_web_x":
            False,

        "direct_nielsen_y":
            False,

        "class_diversity_sufficient":
            "NOT_APPLICABLE",

        "final_role":
            "PROXY_EXPERIMENT",

        "primary_training":
            False,

        "reason":
            "1000 interfaces con X estructural Android y Y rating proxy; no equivalen a X web ni a Y Nielsen directa."
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "x_status":
            "STATIC_WEB_PROXY",

        "y_status":
            "LLM_DIRECT",

        "same_instance_xy":
            True,

        "canonical_web_x":
            False,

        "direct_nielsen_y":
            True,

        "class_diversity_sufficient":
            False,

        "final_role":
            "PIPELINE_VALIDATION",

        "primary_training":
            False,

        "reason":
            "Solo 2 repositorios CONFIRMED_STRONG y ambos tienen targets positivos; no permite clasificación."
    },

    {
        "source_dataset":
            "29042675",

        "x_status":
            "VISUAL_PROTOTYPE_ARTIFACT",

        "y_status":
            "LLM_DIRECT_POSITIVE_EVIDENCE",

        "same_instance_xy":
            True,

        "canonical_web_x":
            False,

        "direct_nielsen_y":
            True,

        "class_diversity_sufficient":
            False,

        "final_role":
            "LABEL_REFERENCE",

        "primary_training":
            False,

        "reason":
            "Las evaluaciones LLM contienen evidencia Nielsen, pero las fuentes son Figma/Behance/SharePoint y los targets recuperados carecen de clase negativa."
    },

    {
        "source_dataset":
            "Dashboards",

        "x_status":
            "BEHAVIOR_LOGS",

        "y_status":
            "EXPERIMENTAL_PROXY_UNJOINED",

        "same_instance_xy":
            False,

        "canonical_web_x":
            False,

        "direct_nielsen_y":
            False,

        "class_diversity_sufficient":
            "UNKNOWN",

        "final_role":
            "X_ONLY_AUXILIARY",

        "primary_training":
            False,

        "reason":
            "No existe un join reproducible entre logs y P+DB+Q."
    },

    {
        "source_dataset":
            "AMUSED",

        "x_status":
            "POTENTIALLY_AVAILABLE",

        "y_status":
            "PROXY_CANDIDATE",

        "same_instance_xy":
            "PENDING_ACCESS",

        "canonical_web_x":
            "UNKNOWN",

        "direct_nielsen_y":
            False,

        "class_diversity_sufficient":
            "UNKNOWN",

        "final_role":
            "PENDING_ACCESS",

        "primary_training":
            False,

        "reason":
            "Acceso al dataset aún no disponible."
    }
])

display(
    source_role_final
)


,source_dataset,x_status,y_status,same_instance_xy,canonical_web_x,direct_nielsen_y,class_diversity_sufficient,final_role,primary_training,reason
0,UICrit_RICO,AVAILABLE_PLATFORM_SPECIFIC,RATING_PROXY,True,False,False,NOT_APPLICABLE,PROXY_EXPERIMENT,False,1000 interfaces con X estructural Android y Y rating proxy; no equivalen a X web ni a Y Nielsen directa.
1,Catching_UX_Flaws,STATIC_WEB_PROXY,LLM_DIRECT,True,False,True,False,PIPELINE_VALIDATION,False,Solo 2 repositorios CONFIRMED_STRONG y ambos tienen targets positivos; no permite clasificación.
2,29042675,VISUAL_PROTOTYPE_ARTIFACT,LLM_DIRECT_POSITIVE_EVIDENCE,True,False,True,False,LABEL_REFERENCE,False,"Las evaluaciones LLM contienen evidencia Nielsen, pero las fuentes son Figma/Behance/SharePoint y los targets recuperados carecen de clase negativa."
3,Dashboards,BEHAVIOR_LOGS,EXPERIMENTAL_PROXY_UNJOINED,False,False,False,UNKNOWN,X_ONLY_AUXILIARY,False,No existe un join reproducible entre logs y P+DB+Q.
4,AMUSED,POTENTIALLY_AVAILABLE,PROXY_CANDIDATE,PENDING_ACCESS,UNKNOWN,False,UNKNOWN,PENDING_ACCESS,False,Acceso al dataset aún no disponible.


## 6. Estado final del modelo Nielsen principal

In [17]:
target_readiness_final = pd.DataFrame([
    {
        "target":
            target,

        "direct_public_training_ready":
            False,

        "reason":
            "Ninguna fuente pública auditada aporta simultáneamente X canónica web, Y Nielsen directa, ambas clases y suficiente número de instancias."
    }
    for target
    in TARGETS
])

display(
    target_readiness_final
)


,target,direct_public_training_ready,reason
0,H1,False,"Ninguna fuente pública auditada aporta simultáneamente X canónica web, Y Nielsen directa, ambas clases y suficiente número de instancias."
1,H4,False,"Ninguna fuente pública auditada aporta simultáneamente X canónica web, Y Nielsen directa, ambas clases y suficiente número de instancias."
2,H5,False,"Ninguna fuente pública auditada aporta simultáneamente X canónica web, Y Nielsen directa, ambas clases y suficiente número de instancias."
3,H6,False,"Ninguna fuente pública auditada aporta simultáneamente X canónica web, Y Nielsen directa, ambas clases y suficiente número de instancias."


## 7. Decisión metodológica

No se realizará todavía un Random Forest Nielsen con las fuentes públicas auditadas.

El siguiente conjunto de entrenamiento deberá satisfacer:

\[
X_{canonical\ web} + Y_{Nielsen}
\]

en la **misma interfaz/ejecución**.

Dado que no se dispone de evaluadores expertos suficientes, la alternativa defendible es crear un conjunto `SILVER` propio:

- X: capturadas directamente por la extensión con las definiciones canónicas;
- Y: evaluación LLM explícita de H1/H4/H5/H6, independiente del cálculo de X;
- procedencia: `LLM_DIRECT_SILVER`;
- nunca presentar esas Y como gold/expert.


In [18]:
training_strategy_final = pd.DataFrame([
    {
        "strategy":
            "PUBLIC_DIRECT_NIELSEN",

        "ready":
            False,

        "recommended":
            False,

        "purpose":
            "Modelo Nielsen principal",

        "reason":
            "No hay fuente pública compatible suficiente."
    },

    {
        "strategy":
            "UICRIT_RICO_PROXY",

        "ready":
            True,

        "recommended":
            True,

        "purpose":
            "Experimento proxy/ablation independiente",

        "reason":
            "Permite estudiar aprendizaje sobre constructos relacionados, sin afirmar clasificación Nielsen web."
    },

    {
        "strategy":
            "OWN_LLM_SILVER_WEB",

        "ready":
            "TO_COLLECT",

        "recommended":
            True,

        "purpose":
            "Fuente principal de desarrollo del clasificador",

        "reason":
            "Permite alinear X canónica web con Y Nielsen silver en la misma instancia."
    },

    {
        "strategy":
            "CATCHING_EXTERNAL_PIPELINE_CHECK",

        "ready":
            True,

        "recommended":
            True,

        "purpose":
            "Validación de trazabilidad/pipeline",

        "reason":
            "Dos casos directos confirmados; no sirven como test estadístico."
    }
])

display(
    training_strategy_final
)


,strategy,ready,recommended,purpose,reason
0,PUBLIC_DIRECT_NIELSEN,False,False,Modelo Nielsen principal,No hay fuente pública compatible suficiente.
1,UICRIT_RICO_PROXY,True,True,Experimento proxy/ablation independiente,"Permite estudiar aprendizaje sobre constructos relacionados, sin afirmar clasificación Nielsen web."
2,OWN_LLM_SILVER_WEB,TO_COLLECT,True,Fuente principal de desarrollo del clasificador,Permite alinear X canónica web con Y Nielsen silver en la misma instancia.
3,CATCHING_EXTERNAL_PIPELINE_CHECK,True,True,Validación de trazabilidad/pipeline,Dos casos directos confirmados; no sirven como test estadístico.


## 8. Requisitos del nuevo conjunto Silver

In [19]:
silver_collection_requirements = pd.DataFrame([
    {
        "requirement":
            "UNIT_OF_OBSERVATION",

        "value":
            "site + page/task + execution",

        "mandatory":
            True,

        "reason":
            "X y Y deben pertenecer exactamente a la misma instancia."
    },

    {
        "requirement":
            "X_SOURCE",

        "value":
            "EXTENSION_CANONICAL_METRICS",

        "mandatory":
            True,

        "reason":
            "Evita incompatibilidad entre Android, screenshots y DOM web."
    },

    {
        "requirement":
            "Y_SOURCE",

        "value":
            "LLM_DIRECT_SILVER",

        "mandatory":
            True,

        "reason":
            "No hay expertos disponibles; mantener procedencia separada."
    },

    {
        "requirement":
            "TARGETS",

        "value":
            "H1_violation | H4_violation | H5_violation | H6_violation",

        "mandatory":
            True,

        "reason":
            "Cuatro clasificadores binarios / salida multilabel."
    },

    {
        "requirement":
            "NEGATIVE_EXAMPLES",

        "value":
            "REQUIRED_FOR_EACH_TARGET",

        "mandatory":
            True,

        "reason":
            "Cada target necesita 0 y 1; ausencia de etiqueta no equivale a 0."
    },

    {
        "requirement":
            "LABEL_INDEPENDENCE",

        "value":
            "LLM must not see derived target rules or X-based thresholds",

        "mandatory":
            True,

        "reason":
            "Evita circularidad y target leakage."
    },

    {
        "requirement":
            "GROUP_SPLIT",

        "value":
            "GROUP_BY_SITE",

        "mandatory":
            True,

        "reason":
            "Una misma aplicación no debe aparecer en train y test."
    },

    {
        "requirement":
            "SOURCE_METADATA",

        "value":
            "source_dataset | site | task | page | label_source | label_level",

        "mandatory":
            True,

        "reason":
            "Trazabilidad y análisis de generalización."
    },

    {
        "requirement":
            "MINIMUM_CLASS_CHECK",

        "value":
            "BOTH_CLASSES_BEFORE_TRAINING",

        "mandatory":
            True,

        "reason":
            "No entrenar targets sin diversidad binaria."
    }
])

display(
    silver_collection_requirements
)


,requirement,value,mandatory,reason
0,UNIT_OF_OBSERVATION,site + page/task + execution,True,X y Y deben pertenecer exactamente a la misma instancia.
1,X_SOURCE,EXTENSION_CANONICAL_METRICS,True,"Evita incompatibilidad entre Android, screenshots y DOM web."
2,Y_SOURCE,LLM_DIRECT_SILVER,True,No hay expertos disponibles; mantener procedencia separada.
3,TARGETS,H1_violation | H4_violation | H5_violation | H6_violation,True,Cuatro clasificadores binarios / salida multilabel.
4,NEGATIVE_EXAMPLES,REQUIRED_FOR_EACH_TARGET,True,Cada target necesita 0 y 1; ausencia de etiqueta no equivale a 0.
5,LABEL_INDEPENDENCE,LLM must not see derived target rules or X-based thresholds,True,Evita circularidad y target leakage.
6,GROUP_SPLIT,GROUP_BY_SITE,True,Una misma aplicación no debe aparecer en train y test.
7,SOURCE_METADATA,source_dataset | site | task | page | label_source | label_level,True,Trazabilidad y análisis de generalización.
8,MINIMUM_CLASS_CHECK,BOTH_CLASSES_BEFORE_TRAINING,True,No entrenar targets sin diversidad binaria.


## 9. Esquema de dataset Silver

In [20]:
silver_schema = pd.DataFrame([
    {
        "column":
            "instance_id",

        "role":
            "ID"
    },

    {
        "column":
            "site_id",

        "role":
            "GROUP"
    },

    {
        "column":
            "page_or_task",

        "role":
            "CONTEXT"
    },

    {
        "column":
            "execution_id",

        "role":
            "CONTEXT"
    },

    {
        "column":
            "<canonical_X_columns>",

        "role":
            "X"
    },

    {
        "column":
            "H1_violation",

        "role":
            "Y"
    },

    {
        "column":
            "H4_violation",

        "role":
            "Y"
    },

    {
        "column":
            "H5_violation",

        "role":
            "Y"
    },

    {
        "column":
            "H6_violation",

        "role":
            "Y"
    },

    {
        "column":
            "label_source",

        "role":
            "PROVENANCE"
    },

    {
        "column":
            "label_level",

        "role":
            "PROVENANCE"
    },

    {
        "column":
            "label_evidence",

        "role":
            "PROVENANCE"
    }
])

display(
    silver_schema
)


,column,role
0,instance_id,ID
1,site_id,GROUP
2,page_or_task,CONTEXT
3,execution_id,CONTEXT
4,<canonical_X_columns>,X
5,H1_violation,Y
6,H4_violation,Y
7,H5_violation,Y
8,H6_violation,Y
9,label_source,PROVENANCE


## 10. Guardar cierre de estrategia

In [21]:
direct_class_audit.to_csv(
    REPORTS_DIR
    / "direct_label_class_audit_final.csv",
    index=False
)

source_role_final.to_csv(
    REPORTS_DIR
    / "source_role_final.csv",
    index=False
)

target_readiness_final.to_csv(
    REPORTS_DIR
    / "target_readiness_final.csv",
    index=False
)

training_strategy_final.to_csv(
    REPORTS_DIR
    / "training_strategy_final.csv",
    index=False
)

silver_collection_requirements.to_csv(
    REPORTS_DIR
    / "silver_collection_requirements.csv",
    index=False
)

silver_schema.to_csv(
    REPORTS_DIR
    / "silver_dataset_schema.csv",
    index=False
)


In [22]:
expected = [
    "direct_label_class_audit_final.csv",
    "source_role_final.csv",
    "target_readiness_final.csv",
    "training_strategy_final.csv",
    "silver_collection_requirements.csv",
    "silver_dataset_schema.csv"
]

for filename in expected:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


direct_label_class_audit_final.csv ✅
source_role_final.csv ✅
target_readiness_final.csv ✅
training_strategy_final.csv ✅
silver_collection_requirements.csv ✅
silver_dataset_schema.csv ✅


## Conclusión

La auditoría pública queda cerrada para fines de entrenamiento directo.

No se debe concatenar RICO, Catching, Dashboards y 29042675 como si fueran un único dataset.

El siguiente notebook debe especificar y preparar la **recolección Silver Web propia** con X canónica de la extensión y Y `LLM_DIRECT_SILVER`, incluyendo balance de clases, protocolo de etiquetado y split por sitio.
